# Assignment 2 — Author co-publication network
This notebook converts the supplied Scopus CSV into an author network for the accompanying D3 visualization. Run every code cell from top to bottom. No third-party Python packages are required.

## What needs correcting in the original notebook
1. `EID` identifies a **paper**, so your nodes and edges form a publication network. The assignment asks for **authors** as nodes and shared publications as links.
2. `fillna(0)` hides missing required values and can mix numbers into text columns. Missing year, author, or affiliation must instead be filtered.
3. `[:-1]` drops a valid ID character if a value has no trailing semicolon. Split on `;`, strip whitespace, and remove empty tokens.
4. `str.contains(author)` performs substring/regular-expression matching rather than exact ID membership.
5. You build a `nodes` list but never call `G.add_nodes_from(nodes)`, so its attributes and isolated nodes are lost.
6. `nx.drawing.draw_spring(G)` draws a plot and returns `None`; it does not return a position dictionary. A position dictionary would come from `nx.spring_layout(G)`.
7. A NetworkX plot does not provide the required D3 simulation, country colors, hover behavior, click tooltip, or controls. Those are implemented in `app.js`.

## Data decisions
- A retained publication must have year, author, and affiliation. We conservatively exclude the entire publication if any listed author has no affiliation, rather than create a partial author team. `cleaning_report.json` lists every exclusion.
- Scopus author IDs identify nodes. Semicolon-delimited affiliation entries align with these IDs; counts are checked before use. Names come from the first two comma-separated components, with suffixes handled separately.
- Countries are recognized from a documented set of country labels observed in this export. An institution without a country remains `Unknown`; its country is not guessed. This set would need updating for a different dataset.
- An author's primary country is the most frequent country across retained affiliation entries; alphabetical order breaks ties. All affiliations and recognized countries are preserved.
- Top 10 means countries ranked by **unique authors' primary countries**, not by paper count. Unknown is excluded from the ranking. All other nodes use `#A9A9A9`.
- One undirected link represents a distinct pair of coauthors. Its weight counts their shared publications. Degree counts distinct coauthors, not the sum of link weights. Single-author publications still create nodes.
- Hover matches any exact affiliation string after case and whitespace normalization. This conservative rule does not merge different departments or variant institution spellings.

## 1. Load, clean, and export
Put `data_scopus.csv` in the same directory as this notebook. In Colab, upload it into `/content/` or `/content/sample_data/`. A Colab file path is not a GitHub URL.


In [1]:
"""Build an undirected AUTHOR co-publication network using Python's standard library."""
import csv
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations

# Use the CSV beside this script/notebook; the second path supports Google Colab.
DATA_PATH = next((p for p in [Path('data_scopus.csv'),
    Path('/content/sample_data/data_scopus.csv'), Path('/content/data_scopus.csv')]
    if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Place data_scopus.csv beside this notebook, or upload it to Colab.')

# Country labels present in this export. Never treat a trailing institution/name as a country.
COUNTRIES = set('United States|Germany|China|United Kingdom|South Korea|Canada|Brazil|Russian Federation|Japan|Spain|France|Australia|Italy|Czech Republic|Austria|Taiwan|Poland|Sweden|Netherlands|India|Portugal|Finland|Switzerland|Greece|Belgium|Ireland|Iran|Tunisia|Norway|Romania|Colombia|Denmark|Philippines|Thailand|Hong Kong|Mexico|Turkey|Singapore|Lithuania|Malaysia|Pakistan|Slovakia|Chile|Saudi Arabia|Kazakhstan|Croatia|Latvia|Serbia|Bulgaria|Trinidad and Tobago|New Zealand|Israel|Egypt|Iraq|Indonesia|Gabon|Hungary|Lebanon|Ecuador|Qatar|Algeria'.split('|'))

def missing(value):
    return not value or value.strip().casefold() in {'nan', 'none', '[no author name available]'}

def parse_authors(row):
    # The semicolon-separated affiliation entries align with the Scopus author IDs.
    # The Authors column cannot safely be split on commas because of Jr. and II.
    ids = [s.strip() for s in row['Author(s) ID'].split(';') if s.strip()]
    entries = [s.strip() for s in row['Authors with affiliations'].split(';') if s.strip()]
    if len(ids) != len(entries) or not ids:
        raise ValueError('Author ID and affiliation entry counts do not match')
    people = []
    for author_id, entry in zip(ids, entries):
        parts = [s.strip() for s in entry.split(',')]
        if len(parts) < 3:
            raise ValueError('At least one author has no affiliation (name-only entry)')
        name = parts[0] + ', ' + parts[1]
        start = 2
        if parts[2].rstrip('.') in {'Jr', 'Sr', 'II', 'III', 'IV'}:
            name += ', ' + parts[2]
            start = 3
        affiliation = ', '.join(parts[start:]).strip(' ,')
        if missing(affiliation):
            raise ValueError('At least one author has no affiliation')
        # Search all components: several entries have an employer after the country.
        countries = sorted({p for p in parts[start:] if p in COUNTRIES})
        people.append((author_id, name, affiliation, countries))
    return people

with DATA_PATH.open(encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))
authors = {}
edge_papers = defaultdict(set)
retained = []
excluded = []
seen = set()
for row in rows:
    eid = row['EID'].strip()
    absent = [key for key in ['Year', 'Authors', 'Authors with affiliations'] if missing(row[key])]
    reason = 'Missing required field(s): ' + ', '.join(absent) if absent else None
    if not reason and not re.fullmatch(r'\d{4}', row['Year'].strip()):
        reason = 'Invalid year'
    if not reason and eid in seen:
        reason = 'Duplicate publication EID'
    if not reason:
        try:
            people = parse_authors(row)
        except ValueError as error:
            reason = str(error)
    if reason:
        excluded.append({'eid': eid, 'reason': reason})
        continue
    seen.add(eid)
    year = int(row['Year'])
    retained.append({'id': eid, 'title': row['Title'], 'year': year})
    for author_id, name, affiliation, countries in people:
        a = authors.setdefault(author_id, {'names': Counter(), 'affiliations': set(),
            'country_votes': Counter(), 'papers': set(), 'years': set()})
        a['names'][name] += 1
        a['affiliations'].add(affiliation)
        a['country_votes'].update(countries)
        a['papers'].add(eid)
        a['years'].add(year)
    # Unique pairs prevent self-loops and duplicate edges within a paper.
    for source, target in combinations(sorted({p[0] for p in people}), 2):
        edge_papers[source, target].add(eid)

def mode(counter):
    return sorted(counter, key=lambda key: (-counter[key], key))[0] if counter else 'Unknown'

links = [{'source': s, 'target': t, 'weight': len(papers), 'publications': sorted(papers)}
         for (s, t), papers in sorted(edge_papers.items())]
degrees = Counter()
for edge in links:
    degrees.update([edge['source'], edge['target']])
nodes = [{'id': aid, 'name': mode(a['names']), 'affiliations': sorted(a['affiliations']),
          'affiliationKeys': sorted({re.sub(r'\s+', ' ', v).strip().casefold() for v in a['affiliations']}),
          'country': mode(a['country_votes']), 'countries': sorted(a['country_votes']),
          'degree': degrees[aid], 'publicationCount': len(a['papers']),
          'publications': sorted(a['papers']), 'years': sorted(a['years'])}
         for aid, a in sorted(authors.items())]
country_counts = Counter(n['country'] for n in nodes if n['country'] != 'Unknown')
top10 = sorted(country_counts, key=lambda c: (-country_counts[c], c))[:10]
metadata = {'inputRecords': len(rows), 'retainedPublications': len(retained),
            'excludedRecords': len(excluded), 'authorCount': len(nodes),
            'linkCount': len(links), 'topCountries': top10,
            'countryCounts': dict(country_counts.most_common()),
            'unknownCountryAuthors': sum(n['country'] == 'Unknown' for n in nodes),
            'degreeRange': [min(n['degree'] for n in nodes), max(n['degree'] for n in nodes)]}
graph = {'nodes': nodes, 'links': links, 'publications': retained, 'metadata': metadata}
Path('author_network.json').write_text(json.dumps(graph, ensure_ascii=False, indent=2), encoding='utf-8')
# The JS wrapper lets index.html also work when opened directly without a server.
Path('author_network.js').write_text('window.AUTHOR_NETWORK = ' + json.dumps(graph, ensure_ascii=False) + ';\n', encoding='utf-8')
Path('cleaning_report.json').write_text(json.dumps({'summary': metadata, 'excluded': excluded}, indent=2), encoding='utf-8')
print(json.dumps(metadata, indent=2))


{
  "inputRecords": 570,
  "retainedPublications": 554,
  "excludedRecords": 16,
  "authorCount": 1535,
  "linkCount": 3025,
  "topCountries": [
    "United States",
    "Germany",
    "China",
    "United Kingdom",
    "South Korea",
    "Japan",
    "Brazil",
    "Canada",
    "Russian Federation",
    "Spain"
  ],
  "countryCounts": {
    "United States": 427,
    "Germany": 137,
    "China": 98,
    "United Kingdom": 78,
    "South Korea": 62,
    "Japan": 57,
    "Brazil": 56,
    "Canada": 54,
    "Russian Federation": 53,
    "Spain": 40,
    "France": 39,
    "Australia": 35,
    "Italy": 28,
    "Austria": 26,
    "Taiwan": 25,
    "Poland": 25,
    "Czech Republic": 23,
    "Sweden": 22,
    "Netherlands": 22,
    "India": 17,
    "Portugal": 17,
    "Greece": 15,
    "Finland": 15,
    "Switzerland": 13,
    "Iran": 11,
    "Norway": 10,
    "Tunisia": 9,
    "Romania": 9,
    "Belgium": 9,
    "Colombia": 8,
    "Ireland": 7,
    "Philippines": 7,
    "Thailand": 6,
    "De

## 2. Validate the exported network
The following checks confirm unique author IDs, unique undirected links, exact shared-publication membership, degree counts, and valid retained attributes.

In [2]:
# Verify graph structure and publication membership, not only file creation.
node_by_id = {n['id']: n for n in nodes}
assert len(node_by_id) == len(nodes)
assert len({(e['source'], e['target']) for e in links}) == len(links)
for e in links:
    assert e['source'] != e['target']
    shared = set(node_by_id[e['source']]['publications']) & set(node_by_id[e['target']]['publications'])
    assert set(e['publications']) == shared
    assert e['weight'] == len(shared)
assert sum(n['degree'] for n in nodes) == 2 * len(links)
assert all(n['affiliations'] and n['years'] and n['name'] for n in nodes)
assert len(rows) == len(retained) + len(excluded)
print('All validation checks passed.')
print('Isolated authors:', sum(n['degree'] == 0 for n in nodes))
print('Excluded publication reasons:', dict(Counter(e['reason'] for e in excluded)))


All validation checks passed.
Isolated authors: 44
Excluded publication reasons: {'At least one author has no affiliation (name-only entry)': 7, 'Missing required field(s): Authors, Authors with affiliations': 9}


## 3. D3 webpage
Open `index.html` from the supplied solution folder. Keep `app.js`, `styles.css`, `author_network.js`, and `vendor/d3.v7.min.js` in their relative locations. The page runs locally without fetching data; the JSON is also supplied as required by the assignment.

Implementation in `app.js`:
- `d3.scaleSqrt().domain(d3.extent(nodes, d => d.degree)).range([3, 12])` determines node radius.
- `d3.scaleOrdinal()` assigns ten country colors; other/unknown countries use `#A9A9A9`.
- `d3.forceSimulation()` uses `forceManyBody`, `forceCollide`, `forceLink`, and centering forces.
- Sliders update charge strength, collision radius multiplier, and link strength, then reheat the simulation.
- Hover keeps matching affiliations at opacity 1 and sets all other nodes to 0.2. Leaving restores every node to opacity 1.
- Clicking opens a tooltip **div** with the author's ID, affiliations, country, degree, and publication count. Nodes also support keyboard focus and Enter/Space.

## 4. GitHub Pages
The files are ready for deployment, but this solution does **not** claim they have been published. Follow the steps in `README.md` using your own GitHub account and repository. Submit the resulting live Pages URL with the notebook if your instructor requests both.

## References
- Supplied assignment brief and `data_scopus.csv`.
- [D3 force simulation](https://d3js.org/d3-force/simulation), [many-body force](https://d3js.org/d3-force/many-body), [collision force](https://d3js.org/d3-force/collide), [link force](https://d3js.org/d3-force/link).
